In [6]:
import cv2
import numpy as np
import random
from PIL import Image

In [2]:
import glob
import os

### Glitch Effect

In [12]:
path = 'anime1.jpg'
folder_path = os.path.join(os.getcwd(), 'photo')
image_paths = glob.glob(os.path.join(folder_path, path))  

In [ ]:
def apply_glitch_effect(img, intensity=15):
    height, width, _ = img.shape
    glitch = img.copy()

    for _ in range(intensity):
        x_start = random.randint(0, width - 1)
        y = random.randint(0, height - 1)
        block_width = random.randint(10, 50)
        shift = random.randint(-30, 30)

        x_end = min(width, x_start + block_width)
        glitch[y:y+1, x_start:x_end] = np.roll(glitch[y:y+1, x_start:x_end], shift, axis=1)

    return glitch

def preserve_faces(original, glitched, face_cascade):
    faces = face_cascade.detectMultiScale(original, 1.3, 5)
    for (x, y, w, h) in faces:
        glitched[y:y+h, x:x+w] = original[y:y+h, x:x+w]
    return glitched

def main():
    output_folder = 'output'
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    image_paths = glob.glob('photo/*.jpg')  

    for img_path in image_paths:
        pil_img = Image.open(img_path)

        img = np.array(pil_img)

        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        if img is None:
            print(f"Can't load image: {img_path}")
            continue
        
        face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
        
        glitched_img = apply_glitch_effect(img, intensity=30)
        
        final_result = preserve_faces(img, glitched_img, face_cascade)

        img_with_title = img.copy()
        cv2.putText(img_with_title, 'Original Image', (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        final_result_with_title = final_result.copy()
        cv2.putText(final_result_with_title, 'Glitch Art with Face Preservation', (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        combined_image = cv2.hconcat([img_with_title, final_result_with_title])

        combined_image_rgb = cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB)
        pil_combined_image = Image.fromarray(combined_image_rgb)

        output_path = os.path.join(output_folder, os.path.basename(img_path))
        pil_combined_image.save(output_path)

        pil_combined_image.show()

In [21]:
if __name__ == '__main__':
    main()

In [ ]:
def edge_detection_and_inversion(img):
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    edges = cv2.Canny(gray_img, 100, 200)

    edges_colored = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    inverted_img = cv2.bitwise_not(img)

    combined_img = cv2.hconcat([edges_colored, inverted_img])

    return combined_img

def main():
    output_folder = 'output'
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    image_paths = glob.glob('photo/*.jpg')  

    for img_path in image_paths:
        pil_img = Image.open(img_path)

        img = np.array(pil_img)

        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        if img is None:
            print(f"Can't load image: {img_path}")
            continue
        
        final_result = edge_detection_and_inversion(img)

        cv2.putText(final_result, 'Edge Detection + Color Inversion', (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        final_result_rgb = cv2.cvtColor(final_result, cv2.COLOR_BGR2RGB)
        pil_result = Image.fromarray(final_result_rgb)

        output_path = os.path.join(output_folder, 'Edge_Detection_and_Color_Inversion_' + os.path.basename(img_path))
        pil_result.save(output_path)

        pil_result.show()

In [23]:
if __name__ == '__main__':
    main()

In [32]:
from PIL import Image, ImageDraw
from scipy.spatial import Voronoi, voronoi_plot_2d
from skimage.color import rgb2gray
from skimage.util import random_noise
import matplotlib.pyplot as plt

In [33]:
output_folder = 'output'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [38]:
filename = 'anime1.jpg'
folder_path = os.path.join(os.getcwd(), 'photo')
image_paths = glob.glob(os.path.join(folder_path, filename))

In [39]:
if not image_paths:
    img_pil = Image.new('RGB', (256, 256), 'gray')
    img_pil.save(os.path.join(folder_path, filename))
    image_path = os.path.join(folder_path, filename)
else:
    image_path = image_paths[0]

In [40]:
img = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

### Polynomial Transformation

In [41]:
def polynomial_transform(img):
    rows, cols, _ = img.shape
    result = np.zeros_like(img)
    for i in range(rows):
        for j in range(cols):
            new_i = int(i + 0.0001 * (j - cols/2)**2)
            new_j = int(j + 0.0001 * (i - rows/2)**2)
            if 0 <= new_i < rows and 0 <= new_j < cols:
                result[new_i, new_j] = img[i, j]
    return result

### Quadtree Decomposition

In [42]:
def quadtree_decomposition(img, threshold=30):
    def split(img, x, y, size):
        region = img[y:y+size, x:x+size]
        if size <= 4 or np.std(region) < threshold:
            cv2.rectangle(img, (x, y), (x+size, y+size), (255, 0, 0), 1)
            return
        half = size // 2
        split(img, x, y, half)
        split(img, x+half, y, half)
        split(img, x, y+half, half)
        split(img, x+half, y+half, half)

    result = img.copy()
    split(result, 0, 0, min(result.shape[:2]))
    return result

### Algorithmic Dithering

In [43]:
def special_dithering(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    dithered = np.zeros_like(gray)
    threshold_map = np.array([[0, 128], [192, 64]])
    for i in range(0, gray.shape[0]):
        for j in range(0, gray.shape[1]):
            d = threshold_map[i % 2, j % 2]
            dithered[i, j] = 255 if gray[i, j] > d else 0
    return cv2.cvtColor(dithered, cv2.COLOR_GRAY2BGR)

### Voronoi Diagram

In [50]:
# def voronoi_effect(img, num_points=50):
#     points = np.random.randint(0, min(img.shape[:2]), size=(num_points, 2))
#     vor = Voronoi(points)
#     result = Image.new('RGB', (img.shape[1], img.shape[0]), 'white')
#     draw = ImageDraw.Draw(result)
#     for region in vor.regions:
#         if not -1 in region and len(region) > 0:
#             polygon = [vor.vertices[i] for i in region]
#             if all(0 <= x < img.shape[1] and 0 <= y < img.shape[0] for x, y in polygon):
#                 color = tuple(np.random.randint(0, 255, size=3))
#                 draw.polygon(polygon, fill=color)
#     return np.array(result)

### Cellular Automata

In [45]:
def cellular_automata(img, iterations=5):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    binary = (gray > 128).astype(np.uint8)
    for _ in range(iterations):
        new = binary.copy()
        for i in range(1, binary.shape[0]-1):
            for j in range(1, binary.shape[1]-1):
                total = np.sum(binary[i-1:i+2, j-1:j+2]) - binary[i, j]
                new[i, j] = 1 if total > 4 else 0
        binary = new
    return cv2.cvtColor(binary * 255, cv2.COLOR_GRAY2BGR)

### Fractal Dimension (Box counting)

In [46]:
def fractal_dimension(img):
    gray = rgb2gray(img)
    threshold = 0.9
    binary = gray < threshold
    sizes = 2**np.arange(1, 6)
    counts = []
    for size in sizes:
        count = 0
        for y in range(0, binary.shape[0], size):
            for x in range(0, binary.shape[1], size):
                if np.any(binary[y:y+size, x:x+size]):
                    count += 1
        counts.append(count)
    fig, ax = plt.subplots()
    ax.plot(np.log(sizes), np.log(counts), 'o-')
    ax.set_title("Fractal Dimension (Log-Log)")
    ax.set_xlabel("log(Box size)")
    ax.set_ylabel("log(Count)")
    plot_path = os.path.join(output_folder, "Fractal_Dimension.png")
    plt.savefig(plot_path)
    plt.close()
    return cv2.imread(plot_path)

### 2D to 3D with Depth Map

In [47]:
def create_depth_map(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    depth = cv2.GaussianBlur(gray, (9, 9), 0)
    depth_colored = cv2.applyColorMap(depth, cv2.COLORMAP_JET)
    return depth_colored

In [51]:
functions = {
    "Polynomial_Transform": polynomial_transform,
    "Quadtree_Decomposition": quadtree_decomposition,
    "Algorithmic_Dithering": special_dithering,
    # "Voronoi_Effect": voronoi_effect,
    "Cellular_Automata": cellular_automata,
    "Fractal_Dimension": fractal_dimension,
    "Depth_Map_3D": create_depth_map
}

In [52]:
for name, func in functions.items():
    result = func(img.copy())
    if isinstance(result, np.ndarray):
        result_img = Image.fromarray(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    else:
        result_img = Image.fromarray(result)
    result_img.show(title=name.replace("_", " "))
    result_img.save(os.path.join(output_folder, f"{name}.png"))